# EOA Framework — 02. Live Evaluation with Google Gemini API

This notebook executes the **EOA (Explanation-Oriented Agents)** architecture live against the **Google Gemini API**.

**Automatic `.env` Discovery:**
`ExplanationLayer.default_layer()` and `ExplanationLayer()` automatically read `GEMINI_API_KEY` and optional `GEMINI_MODEL` from the environment.

## 1. Setup & Auto-Load `.env` Credentials

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root/src is in Python path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR / "src"))

# Load .env file from project root into os.environ if present
env_file = ROOT_DIR / ".env"
if env_file.is_file():
    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip("'\""))

import eoa
from eoa import (
    ExplanationLayer,
    SpecializedAgent,
    SQLiteSessionStore,
)

# Initialize layer and create the 4 canonical default agents (CMA, AA, EA, VA)
layer = ExplanationLayer(provider="gemini")
layer.create_default_agents()
print(f"EOA v{eoa.__version__} initialized with live model: {layer.llm.model}")

## 2. Live Scenario 1: Zero-ML Mode (Progressive Ramp on Hydraulic Domain)

Before plugging in predictive ML models or SHAP values, we test **Zero-ML Mode** using strictly the hydraulic domain context (`variables`, `thresholds`, and `POP_Resfriamento_04`).
- `ArtifactsAgent` automatically bypasses LLM execution (0 tokens spent on AA).
- `ContextManagerAgent`, `ExplainerAgent`, and `ValidatorAgent` answer and audit strictly against the operational manual.

In [ ]:
incident_402 = {
    "prediction": {
        "model_name": "XGBoost_CoolerCondition_v2",
        "target_variable": "Condição do Resfriador Hidráulico",
        "predicted_value": "Eficiência Reduzida (Nível Crítico)",
        "confidence_or_probability": 0.86,
        "status_label": "Operação Degradada / Alerta Crítico",
    },
    "xai": {
        "method": "SHAP",
        "feature_attributions": {
            "PS1_Pressao_Entrada": 0.54,
            "EPS1_Potencia_Motor": 0.32,
            "TS2_Temp_Trocador": 0.14,
        },
    },
    "context": {
        "variables": {
            "PS1_Pressao_Entrada": "Bar, faixa normal de operação [2.0, 4.0] bar",
            "EPS1_Potencia_Motor": "kW, faixa normal de operação [1.5, 2.8] kW",
        },
        "thresholds": {"PS1_max": "4.0 bar", "EPS1_max": "3.0 kW"},
        "documents": {
            "POP_Resfriamento_04": "Seção 4: Se PS1 exceder 4.0 bar acompanhado por EPS1 > 3.0 kW, há provável cavitação no filtro de sucção. Reduzir carga hidráulica imediatamente."
        },
    },
}

In [ ]:
# Zero-ML Execution: Only domain context and user_query (prediction=None, xai=None)
zero_ml_res = layer.explain(
    context=incident_402["context"],
    user_query="Segundo o procedimento POP_Resfriamento_04, o que significa quando PS1 passa de 4.0 bar junto com EPS1 acima de 3.0 kW e qual ação o operador deve tomar?",
)

print("=== MODO ZERO-ML (DOMÍNIO HIDRÁULICO PURO) ===\n")
print(zero_ml_res.narrative)
print("\n" + "=" * 70)
print(f"Nota da Auditoria (ValidatorAgent): {zero_ml_res.validation_score}/10.0 | Aprovado: {zero_ml_res.validation_passed}")
for step in zero_ml_res.execution_trace:
    print(f"[{step.agent_name:<16}] {step.execution_time_ms:>7.1f} ms | {step.summary}")
print("=" * 70)

## 3. Live Scenario 2: Full ML Diagnostic Mode (Prediction + SHAP + Domain Context)

In [ ]:
response = layer.explain(
    prediction=incident_402["prediction"],
    xai=incident_402["xai"],
    context=incident_402["context"],
    user_query="Explique por que este alerta disparou e qual ação imediata tomar em campo.",
)

print("=== NARRATIVA REAL GERADA PELO GEMINI ===\n")
print(response.narrative)
print("\n" + "=" * 70)
print(f"Nota da Auditoria (ValidatorAgent): {response.validation_score}/10.0")
print(f"Aprovado no Quality Gate: {response.validation_passed} | Tentativas: {response.attempts_used}")
print("=" * 70)

In [ ]:
for step in response.execution_trace:
    print(f"[{step.agent_name:<16}] {step.execution_time_ms:>7.1f} ms | status={step.status:<10} | {step.summary}")

## 4. Live Scenario 3: Multi-Turn Conversation with SQLite Persistence

Here we initialize a session-backed layer in 1 line using `ExplanationLayer.default_layer(session_store=store)`.

In [ ]:
store = SQLiteSessionStore(db_path=ROOT_DIR / "examples" / "live_gemini_sessions.db")
chat_layer = ExplanationLayer.default_layer(session_store=store)
session_id = "sessao_real_turbina_01"
chat_layer.clear_session(session_id)

# Turn 1: Initial diagnosis
res1 = chat_layer.explain(
    prediction=incident_402["prediction"],
    xai=incident_402["xai"],
    context=incident_402["context"],
    user_query="Resuma o problema detectado no resfriador.",
    session_id=session_id,
)
print("[TURNO 1]:\n", res1.narrative)

# Turn 2: Operator follow-up question (inherits session artifacts automatically!)
res2 = chat_layer.explain(
    user_query="Se não pudermos parar a bomba agora, qual o risco para o motor elétrico (EPS1)?",
    session_id=session_id,
)
print("\n[TURNO 2 - FOLLOW-UP]:\n", res2.narrative)
print(f"\nScore Turno 2: {res2.validation_score}/10.0 | Mensagens no banco: {len(chat_layer.get_history(session_id))}")

In [ ]:
# Turn 3: Follow-up query executed asynchronously (await aexplain)
res3 = await chat_layer.aexplain(
    user_query="Me explica as variáveis que estamos tratando e o que elas representam, você tem alguma tool associada?",
    session_id=session_id,
)

print(f"\n[TURNO 3 - FOLLOW-UP (Async)] (Score: {res3.validation_score}/10.0):\n", res3.narrative)

## 5. Live Scenario 4: Prompt Override & Objective Follow-Up Responses

In this scenario, we use `chat_layer.override_agent_prompt` to customize the `ExplainerAgent` prompt for concise and direct answers without repeating the entire incident diagnosis.

In [ ]:
# Sobrescreve o prompt do Explainer para respostas concisas no chat
chat_layer.override_agent_prompt(
    name="explainer",
    prompt="""
Responda de forma extremamente objetiva e técnica à dúvida do operador, sem rodeios e sem encher linguiça.

Dúvida do operador:
{user_query}
""",
)

# Inspeciona a camada (mostra "Inline (str)" para o Explainer)
chat_layer.inspect()

# Turno 4: Pergunta conceitual de follow-up
res4 = chat_layer.explain(
    user_query="O que é um motor de pressão?",
    session_id=session_id,
)

print(f"\n[TURNO 4 - Pós reescrita no prompt] (Score: {res4.validation_score}/10.0):\n", res4.narrative)

## 6. Live Scenario 5: Specialized Agent with Autonomous Tool Selection & Extraction

Here we register a `SpecializedAgent` (`math`) with a custom Python tool function (`soma_fake(a: int, b: int) -> int`).

Notice that we do **not** need to pass `query_params={"a": 2, "b": 3}` manually: the `math` agent inspects its tool catalog (`parameters_schema`), decides via reasoning that `soma_fake` is needed for the user's question, extracts `a=2` and `b=3` directly from the natural language query, executes the Python function (`2 * 3 = 6`), and forwards the specialist insight to the `ExplainerAgent`.

In [ ]:
# 1. Restaura o prompt do Explainer para sintetizar pareceres dos especialistas
chat_layer.reset_agent_prompt("explainer")


# 2. Função customizada Python com parâmetros tipados
def soma_fake(a: int, b: int) -> int:
    """Multiplica os fatores a e b simulando uma soma fake customizada."""
    return a * b


# 3. Cria o especialista com o placeholder {tool_outputs}
matematico = SpecializedAgent(
    name="math",
    role="Especialista em Matemática Fake",
    prompt="""
Você é um especialista em matemática fake.
A ferramenta soma_fake executou e calculou o seguinte resultado numérico: {tool_outputs}
Explique o resultado obtido e o conceito de cálculo fake para a dúvida: {user_query}
""",
)

# 4. Registra a ferramenta no especialista e o especialista na camada
matematico.register_tool(soma_fake)
chat_layer.register_agent(matematico)

# Inspeciona a camada (mostra math ativo com 1 Tool e explainer Built-in)
chat_layer.inspect()

# 5. Executa apenas com a pergunta em linguagem natural (o agente extrai a=2 e b=3 autonomamente!):
res5 = chat_layer.explain(
    user_query="Me explica o que é um cálculo fake e apresente o resultado da soma fake para 2 e 3.",
    session_id=session_id,
)

print(f"\n[TURNO 5 - Especialista + Tool Autônoma (Score: {res5.validation_score}/10.0)]:\n", res5.narrative)

## 7. Inspecting Full Session History (`chat_layer.get_history`)

In [ ]:
history = chat_layer.get_history(session_id=session_id)

for msg in history:
    print(f"[{msg.role.upper()}]: {msg.content}\n" + "-" * 60)